# Anthony Wolfe's KNN Model

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [7]:
TRAIN_CSV = "C:\\Users\\wolfe\\OneDrive\\Desktop\\311-projects\\project3\\data\\train_biomass.csv" 
VAL_CSV   = "C:\\Users\\wolfe\\OneDrive\\Desktop\\311-projects\\project3\\data\\validate_biomass.csv"
TEST_CSV  = "C:\\Users\\wolfe\\OneDrive\\Desktop\\311-projects\\project3\\data\\test_biomass.csv"

def weighted_distance_2d(p, q, w):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    dx1 = p[0] - q[0]
    dx2 = p[1] - q[1]

    w1 = w
    w2 = 1.0 - w

    return np.sqrt((w1 * dx1) ** 2 + (w2 * dx2) ** 2)


def knn_predict_one(query_row, train_df, k, w):
    species = query_row["SPCD"]
    subset = train_df[train_df["SPCD"] == species]

    if subset.empty:
        subset = train_df

    q = np.array([query_row["DO_BH"], query_row["HT_TOT"]], dtype=float)
    xs = subset[["DO_BH", "HT_TOT"]].to_numpy(dtype=float)

    distances = np.array(
        [weighted_distance_2d(q, xs[i], w) for i in range(len(xs))]
    )

    k_eff = int(min(k, len(distances)))

    nn_idx = np.argpartition(distances, k_eff - 1)[:k_eff]

    y_neighbors = subset.iloc[nn_idx]["TT_DW_CRM"].to_numpy(dtype=float)
    return float(y_neighbors.mean())


def knn_predict_dataset(df, train_df, k, w):
    preds = []
    for _, row in df.iterrows():
        y_hat = knn_predict_one(row, train_df, k, w)
        preds.append(y_hat)
    return np.array(preds, dtype=float)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean((y_true - y_pred) ** 2)**(1/2))


def tune_knn(train_df, val_df, k_values, w_values):
    results = []
    y_val_true = val_df["TT_DW_CRM"].to_numpy(dtype=float)

    for k in k_values:
        for w in w_values:
            val_preds = knn_predict_dataset(val_df, train_df, k, w)
            val_mse = rmse(y_val_true, val_preds)
            results.append({"k": int(k), "w": float(w), "val_mse": val_mse})
            print(f"[tuning] k={k:2d}, w={w:.2f}, val RMSE={val_mse:.4f}")

    results_df = pd.DataFrame(results)
    best_row = results_df.loc[results_df["val_mse"].idxmin()]
    best_k = int(best_row["k"])
    best_w = float(best_row["w"])
    return best_k, best_w, results_df


def main():
    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    test_df  = pd.read_csv(TEST_CSV)

    required_cols = {"SPCD", "DO_BH", "HT_TOT", "TT_DW_CRM"}
    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        missing = required_cols.difference(df.columns)
        if missing:
            raise ValueError(
                f"{name} dataset is missing columns: {sorted(missing)}"
            )

    print(f"Loaded train: {len(train_df)} rows")
    print(f"Loaded val  : {len(val_df)} rows")
    print(f"Loaded test : {len(test_df)} rows")

    k_values = list(range(1, 20, 2))
    w_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]

    print("\n=== Tuning KNN on validation set ===")
    best_k, best_w, results_df = tune_knn(train_df, val_df, k_values, w_values)

    print("\n=== Best hyperparameters from validation ===")
    print(f"best k = {best_k}")
    print(f"best w = {best_w:.4f}")

    train_val_df = pd.concat([train_df, val_df], ignore_index=True)

    print("\n=== Evaluating on test set ===")
    y_test_true = test_df["TT_DW_CRM"].to_numpy(dtype=float)
    y_test_pred = knn_predict_dataset(test_df, train_val_df, best_k, best_w)
    test_mse = rmse(y_test_true, y_test_pred)

    print(f"Test RMSE with k={best_k}, w={best_w:.4f}: {test_mse:.4f}")


if __name__ == "__main__":
    main()


Loaded train: 45693 rows
Loaded val  : 45693 rows
Loaded test : 45693 rows

=== Tuning KNN on validation set ===
[tuning] k= 1, w=0.10, val MSE=5878.9284
[tuning] k= 1, w=0.20, val MSE=5005.6960
[tuning] k= 1, w=0.30, val MSE=3102.5195
[tuning] k= 1, w=0.40, val MSE=2947.5623
[tuning] k= 1, w=0.50, val MSE=2608.2416
[tuning] k= 1, w=0.60, val MSE=2589.2139
[tuning] k= 1, w=0.70, val MSE=2584.4033
[tuning] k= 1, w=0.80, val MSE=2948.8107
[tuning] k= 1, w=0.90, val MSE=3129.6528
[tuning] k= 1, w=1.00, val MSE=3302.9992
[tuning] k= 3, w=0.10, val MSE=7876.9281
[tuning] k= 3, w=0.20, val MSE=5565.9355
[tuning] k= 3, w=0.30, val MSE=4870.0745
[tuning] k= 3, w=0.40, val MSE=4595.9128
[tuning] k= 3, w=0.50, val MSE=4204.2125
[tuning] k= 3, w=0.60, val MSE=4026.9612
[tuning] k= 3, w=0.70, val MSE=3934.5659
[tuning] k= 3, w=0.80, val MSE=3913.4853
[tuning] k= 3, w=0.90, val MSE=3944.7568
[tuning] k= 3, w=1.00, val MSE=3983.5189
[tuning] k= 5, w=0.10, val MSE=8828.5530
[tuning] k= 5, w=0.20, val